# Logos Pretraining — Colab TPU v6e

End-to-end pretraining of a single **Logos** transformer on a **TPU v6e** runtime with the XLA training driver.

**Architecture (Logos)**
- **KDA + Snapshot Retrieval** sub-quadratic attention with **MLA-compressed snapshots**
- **Local Sliding-Window Attention** every 4 layers (Samba-style)
- **Block Attention Residuals** (depth-wise softmax over completed blocks)
- **Three-section** Entry → Body (looped 3×) → Exit
- **Mixture-of-Experts** body: shared + sparse experts, top-k routing, with **cross-loop expert diversity**
- **Multi-Token Prediction** heads (DeepSeek-V3 style), shared embedding + LM head
- **Attention sink**, **QK norm**, **partial RoPE**

**Training pipeline**
- **PyTorch/XLA** launcher + `MpDeviceLoader` + XLA gradient all-reduce
- **Muon** (2D Linears) + **AdamW** (1D scalars + embedding) split with differentiated base lrs
- **WSD** schedule (warmup → stable → linear decay), Muon-only **momentum ramp** + **WD anneal**
- **EMA disabled** by default to reduce memory and validation overhead
- **Rolling-N** checkpoints (best/final preserved)
- **bf16 AMP on XLA** and **streaming** FineWeb-Edu corpus

This notebook is a thin wrapper: it builds an `argparse.Namespace` of training flags and hands it to `scripts.train_xla.main(args)`. There's no duplicated training loop — model construction, optimizer split, W&B logging, MoE bias updates, sampling, and checkpointing live in `train_xla.py` / `train.py` helpers.

> **Runtime:** set Colab to **TPU v6e** before running: `Runtime → Change runtime type → TPU v6e`. Colab currently exposes `TPUv6e-1`, so this notebook uses XLA single-process launch.
> **Prerequisite:** make sure your fork of `Logos` on GitHub has the latest master branch. The clone cell below does `git fetch && git reset --hard origin/master` so subsequent runs pick up new commits.


## 1. TPU sanity check

In [ ]:
import os, importlib.util
os.environ.setdefault('PJRT_DEVICE', 'TPU')
print('PJRT_DEVICE =', os.environ['PJRT_DEVICE'])
print('torch_xla installed:', importlib.util.find_spec('torch_xla') is not None)
print('Expected runtime: Colab TPU v6e')


## 2. Install dependencies

Colab TPU images usually include a matching `torch` / `torch_xla` pair. The next cell checks that pair and installs the regular Logos Python dependencies. If `torch_xla` is missing, install a matching PyTorch/XLA build for the runtime, restart the notebook, and rerun from the top.


In [ ]:
import os, sys, subprocess, importlib.util
os.environ.setdefault('PJRT_DEVICE', 'TPU')

import torch
print(f'torch: {torch.__version__}')

if importlib.util.find_spec('torch_xla') is None:
    raise RuntimeError(
        'torch_xla is not installed in this runtime. Switch Colab to TPU v6e, '
        'or install a torch_xla wheel that exactly matches torch '
        f'{torch.__version__}, then restart the runtime.'
    )

import torch_xla
print(f'torch_xla: {getattr(torch_xla, "__version__", "<unknown>")}')


In [ ]:
!pip install -q -U datasets transformers tiktoken einops tqdm


## 3. Get the Logos repo

Replace the URL with your fork if you have one. The cell does a `git fetch && git reset --hard origin/master` if the path already exists, so re-running picks up new commits.

In [ ]:
%cd /content/
import os, pathlib, subprocess
REPO_URL  = 'https://github.com/Rorical/Logos.git'
REPO_PATH = '/content/Logos'

if pathlib.Path(REPO_PATH).exists():
    print('Repo exists - fetching + hard-resetting to origin/master')
    subprocess.check_call(['git', '-C', REPO_PATH, 'fetch', '--all', '--prune'])
    subprocess.check_call(['git', '-C', REPO_PATH, 'reset', '--hard', 'origin/master'])
else:
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_PATH])

print('HEAD:')
subprocess.check_call(['git', '-C', REPO_PATH, 'log', '--oneline', '-1'])
%cd /content/Logos


## 4. Mount Google Drive (optional but recommended)

Checkpoints land in `MyDrive/logos-pretraining-tpu-v6e/` so they survive runtime restarts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib

CHECKPOINT_DIR = '/content/drive/MyDrive/logos-pretraining-tpu-v6e'
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print('Checkpoints will go to', CHECKPOINT_DIR)


## 5. Imports + TPU/XLA info

In [ ]:
import os, sys, torch
os.environ.setdefault('PJRT_DEVICE', 'TPU')
import torch_xla
sys.path.insert(0, '/content/Logos')

from scripts.train_xla import build_arg_parser, main

print(f'PyTorch  : {torch.__version__}')
print(f'torch_xla: {getattr(torch_xla, "__version__", "<unknown>")}')
print(f'PJRT     : {os.environ.get("PJRT_DEVICE")}')
print('Device allocation happens inside scripts.train_xla.main() via torch_xla.launch().')


## 6. API compatibility check

Fail fast with an actionable message if the cloned repo is older than the TPU/XLA training work.


In [ ]:
from models.logos import LogosConfig
import scripts.train as train_helpers
import scripts.train_xla as train_xla

required = []
for name in ('build_arg_parser', 'main', 'run_step_training_xla',
             'evaluate_xla', 'save_checkpoint_xla'):
    if not hasattr(train_xla, name):
        required.append(f'scripts.train_xla.{name}')
for name in ('PackedStream', 'build_streaming_loaders',
             'split_param_groups', 'build_optimizer_and_scheduler'):
    if not hasattr(train_helpers, name):
        required.append(f'scripts.train.{name}')

if required:
    raise RuntimeError(
        'Cloned repo is missing required APIs: ' + ', '.join(required) +
        '\nRe-run the clone cell to fetch latest origin/master.'
    )
print('API compatibility OK.')


## Scale & memory budget

**TPU v6e target.** Colab currently exposes `TPUv6e-1`, so this notebook passes `--xla-single-process` and trains on one XLA device. If you move the same notebook to a multi-device TPU VM, remove that flag to let `torch_xla.launch()` spawn across the full slice. Data-parallel XLA replicas each hold a full copy of the model, optimizer state and activations; they increase throughput, but do not shard model memory.

**Training horizon.** `--total-tokens 10B` derives `total_steps` automatically from `world_size × batch_size × max_length`. With `TPUv6e-1` and `--xla-single-process`, `world_size=1`. On larger TPU slices, removing `--xla-single-process` lets the token budget account for the larger XLA world size automatically.

**Starting v6e profile.** This fork starts with `bs=1` per replica, `seq=2048`, gradient checkpointing, chunked LM-head CE, and a moderately sized Logos config. Logos uses `torch_xla.utils.checkpoint` on XLA so checkpointing avoids PyTorch 2.9's `torch.xla` device-module path.

**XLA compile behavior.** Do not pass `--compile`; XLA traces and compiles device graphs lazily. The first steps and the first eval/sample at a new shape can take longer while XLA compiles.

**If you OOM:** first drop `--max-length` to 1024, then `--expert-d-ff` to 384, then `--d-model` to 768 with `--num-heads 12`. `--total-tokens` keeps the training budget fixed, so smaller per-step batches add steps rather than silently shrinking the corpus.


## 7. Build the training Namespace

This is the single source of configuration for the run. Everything below is a flag from `scripts/train_xla.py`'s argument parser — no model build / optimizer / loop code lives in this notebook.


In [ ]:
parser = build_arg_parser()

# TPU v6e Colab run. Every CLI knob from scripts/train_xla.py that
# matters for this run is set explicitly below. Torch DDP and
# torch.compile-specific flags are intentionally omitted: XLA handles
# replica launch, gradient all-reduce, host-to-device loading, and lazy
# compilation inside scripts/train_xla.py.
cli = [
    '--model', 'logos',

    # ---- Streaming corpus ----
    '--streaming',
    '--dataset',         'HuggingFaceFW/fineweb-edu',
    '--dataset-config',  'sample-10BT',
    '--text-column',     'text',
    '--val-docs',        '200',
    '--tiktoken-encoding', 'cl100k_base',  # cl100k_base | o200k_base

    # ---- Training horizon ----
    '--total-tokens', '10B',     # derived as ceil(10e9 / (xla_world_size * bs * seq))
    '--batch-size',   '1',       # per XLA replica; increase only after memory is confirmed
    '--max-length',   '2048',
    '--log-every',    '50',
    '--eval-every',   '10000',
    '--save-every',   '5000',
    '--sample-every', '20000',
    '--keep-last-n',  '5',

    # ---- DataLoader / XLA host-to-device pipeline ----
    '--num-workers',     '2',
    '--prefetch-factor', '2',
    '--xla-batches-per-execution', '1',
    '--xla-single-process',       # required for Colab TPUv6e-1; remove on multi-device TPU VMs

    # ---- Mixed precision + activation memory ----
    '--bf16',
    '--gradient-checkpointing',
    '--ckpt-granularity', 'per-block',  # 'per-block' = lowest backward peak (allows largest --batch-size); 'per-loop' = better compile fusion when memory has headroom

    # ---- Logos architecture ----
    '--d-model',          '1024',
    '--num-heads',        '16',
    '--head-dim',         '64',
    '--d-ff',             '2730',
    '--num-entry-layers', '2',
    '--num-body-layers',  '4',
    '--num-exit-layers',  '2',
    '--num-loops',        '3',
    '--dropout',          '0.0',
    '--norm-eps',         '1e-6',
    '--body-gate-init-std', '0.0',

    # KDA + retrieval
    '--chunk-size',          '128',
    '--conv-size',           '4',
    '--snapshot-interval',   '512',
    '--snapshot-latent-dim', '128',
    '--mem-top-k',           '16',
    '--mem-head-dim',        '64',

    # RoPE
    '--rope-scaling-type',   'none',  # none | ntk | yarn
    '--rope-scaling-factor', '1.0',
    '--yarn-beta-fast',      '32.0',
    '--yarn-beta-slow',      '1.0',
    # '--rope-original-max-position', '2048',  # set when using ntk/yarn

    # SWA
    '--swa-window', '256',
    '--swa-every',  '4',
    '--swa-offset', '3',

    # MoE
    '--use-moe',
    '--num-shared-experts',   '2',
    '--num-sparse-experts',   '32',
    '--top-k',                '4',
    '--expert-d-ff',          '512',
    '--moe-diversity-factor', '0.5',
    '--bias-update-rate',     '0.01',
    '--moe-log-every',        '1000',
    # Memory-efficient LM head: chunked tied-CE.
    '--lm-head-chunk-size', '512',

    # ---- Optimizer (Muon + AdamW) ----
    '--muon',
    '--muon-schedule-hyperparams',
    '--lr',           '0.004',
    '--embed-lr',     '0.2',
    '--muon-lr',      '0.02',
    '--weight-decay', '0.1',
    '--grad-clip',    '1.0',

    # ---- WSD schedule + Muon hyperparam ramp ----
    '--scheduler',    'wsd',
    '--warmup-steps', '3500',
    '--decay-steps',  '70000',
    '--decay-frac',   '0.2',
    '--muon-momentum-start',     '0.85',
    '--muon-momentum-mid',       '0.90',
    '--muon-momentum-end',       '0.95',
    '--muon-momentum-warmup-1',  '150',
    '--muon-momentum-warmup-2',  '300',
    '--muon-wd-start',           '0.2',
    '--muon-wd-end',             '0.0',

    # ---- EMA shadow weights ----
    '--ema-decay', '0.0',  # disabled by default; set 0.999 or 0.9999 to enable

    # ---- Sampling ----
    '--sample-prompt',      'In a recent study, researchers found that',
    '--sample-max-tokens',  '80',
    '--sample-temperature', '0.8',

    # ---- Output ----
    '--save-dir', CHECKPOINT_DIR,
    '--seed',     '42',

    # ---- Weights & Biases (uncomment to enable; requires `wandb login`) ----
    # '--wandb',
    # '--wandb-project',  'logos-pretrain',
    # '--wandb-entity',   'YOUR_WANDB_ENTITY',
    # '--wandb-run-name', 'logos-tpu-v6e-fineweb10BT-colab',
    # '--wandb-tags',     'logos', 'fineweb-edu', 'colab', 'tpu-v6e', 'xla',
    # '--wandb-mode',     'online',  # online | offline | disabled
]

args = parser.parse_args(cli)
print('Run config:')
for k, v in sorted(vars(args).items()):
    print(f'  {k:<28} {v}')


## 8. Train

Single call. `main(args)` does:
1. Launch a single TPU/XLA worker with `torch_xla.launch(..., debug_single_process=True)` for Colab `TPUv6e-1`.
2. Build tokenizer + streaming dataloaders (`build_streaming_loaders`).
3. Wrap dataloaders with `MpDeviceLoader` for host-to-device transfer.
4. Build the Logos model and broadcast rank-0 initial weights across replicas.
5. Three-way Muon/AdamW split (`split_param_groups`) + WSD scheduler + Muon momentum/wd ramp.
6. EMA is disabled by default (`--ema-decay 0.0`) to reduce memory and validation overhead.
7. Step-bounded XLA loop: forward → backward → XLA gradient reduce/optimizer step → sched → muon_hp → MoE bias update, with periodic eval / sample / checkpoint, rolling-N pruning.
8. Final checkpoint + `history.json` written to `--save-dir`.


In [ ]:
main(args)

## 9. (Optional) Sample from the final model after training

Loads the final checkpoint and generates a continuation on a single XLA device. If you later enable EMA, this cell will use the EMA weights when present; otherwise it uses the live final weights.


In [ ]:
import os, torch
from pathlib import Path
from models.logos import LogosConfig, LogosTransformer
from utils.tokenizer import TiktokenTokenizer
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
import torch_xla
import torch_xla.core.xla_model as xm

os.environ.setdefault('PJRT_DEVICE', 'TPU')
device = torch_xla.device() if hasattr(torch_xla, 'device') else xm.xla_device()

ckpt = torch.load(Path(CHECKPOINT_DIR) / 'checkpoint_final.pt',
                  map_location='cpu', weights_only=False)

# Rebuild config from the args we passed (we still have `args` from cell 7).
cfg_kwargs = {k: v for k, v in vars(args).items() if k in LogosConfig.__dataclass_fields__}
cfg = LogosConfig(**{**cfg_kwargs, 'vocab_size': TiktokenTokenizer(args.tiktoken_encoding).vocab_size})

model = LogosTransformer(cfg).to(device)
model.load_state_dict(ckpt['model_state_dict'])

if ckpt.get('ema_state_dict'):
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(args.ema_decay), use_buffers=True).to(device)
    ema.load_state_dict(ckpt['ema_state_dict'])
    target = ema.module
    print('Using EMA weights')
else:
    target = model
    print('No EMA in checkpoint - using live weights')

tok = TiktokenTokenizer(args.tiktoken_encoding)
prompt = 'The history of artificial intelligence began'
ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
target.eval()
with torch.no_grad(), torch.autocast('xla', dtype=torch.bfloat16):
    out = target.generate(ids, max_new_tokens=200, temperature=0.7, top_k=40)
xm.mark_step()
print(tok.decode(out[0].cpu().tolist()))


## 10. (Optional) Plot the loss curve

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

with (Path(CHECKPOINT_DIR) / 'history.json').open() as f:
    hist = json.load(f)

events = hist['history']
val_pts = [(h['step'], h['val']['loss']) for h in events if h.get('val') and h['val']['loss'] != float('inf')]
ema_pts = [(h['step'], h['ema_val']['loss']) for h in events if h.get('ema_val') and h['ema_val']['loss'] != float('inf')]

plt.figure(figsize=(10, 5))
if val_pts: plt.plot(*zip(*val_pts), 'o-', label='val', linewidth=2)
if ema_pts: plt.plot(*zip(*ema_pts), 's-', label='ema val', linewidth=2)
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Logos pretraining'); plt.show()
